In [19]:
import pandas as pd

def preprocess(path="dataset.csv"):

    # Load dataset
    data = pd.read_csv(path)

    data['date'] = pd.to_datetime(data['date'], format='%d-%m-%Y')

    data['time'] = pd.to_datetime(data['time'], format='%H:%M:%S')

    # Remove extra spaces in column names
    data.columns = data.columns.str.strip()

    # Extract hour for time analysis
    data['hour'] = data['time'].dt.hour

    # Ensure numeric columns are numeric
    numeric_cols = ['quantity','unit_price','total_price']

    for col in numeric_cols:
        data[col] = pd.to_numeric(data[col], errors='coerce')

    # Remove rows with missing important values
    data = data.dropna(subset=['transaction_id','item','quantity'])

    # Remove duplicate rows if any
    data = data.drop_duplicates()

    # Standardize text columns
    text_cols = ['shop','item','category','payment_method','customer_type']

    for col in text_cols:
        data[col] = data[col].astype(str).str.strip()

    return data

In [20]:
def get_top_products(data, n=10):

    top_products = (
        data.groupby("item")["quantity"]
        .sum()
        .sort_values(ascending=False)
        .head(n)
    )

    return top_products.reset_index().to_dict(orient="records")

In [21]:
def get_top_products_by_shop(data, shop, n=5):

    shop_products = (
        data.groupby(["shop", "item"])["quantity"]
        .sum()
        .reset_index()
    )

    shop_data = shop_products[shop_products["shop"] == shop]

    top_items = shop_data.sort_values("quantity", ascending=False).head(n)

    return top_items.to_dict(orient="records")

In [22]:
def get_trending_products(data, n=10):

    # Ensure date is datetime
    data['date'] = pd.to_datetime(data['date'], format='%d-%m-%Y')

    # Step 1: Daily sales
    daily_sales = (
        data.groupby(["date", "item"])["quantity"]
        .sum()
        .reset_index()
    )

    # Step 2: Pivot table
    trend = daily_sales.pivot(
        index="date",
        columns="item",
        values="quantity"
    ).fillna(0)

    # Step 3: Sort dates (IMPORTANT)
    trend = trend.sort_index()

    # Step 4: Growth calculation
    trend_growth = trend.diff().mean().sort_values(ascending=False)

    # Step 5: Top N
    trending = trend_growth.head(n)

    # Step 6: Clean output (FIXED)
    return [
        {"item": item, "trend_score": float(score)}
        for item, score in trending.items()
    ]

In [23]:
def get_peak_hours(data):

    hourly_sales = data.groupby("hour")["quantity"].sum()

    return hourly_sales.reset_index().to_dict(orient="records")

In [24]:
def get_vendor_recommendations(data, shop, n=5):

    recommend = (
        data.groupby(["shop", "item"])["quantity"]
        .sum()
        .reset_index()
    )

    shop_data = recommend[recommend["shop"] == shop]

    top_items = shop_data.sort_values("quantity", ascending=False).head(n)

    return top_items.to_dict(orient="records")

In [25]:
data = preprocess("dataset.csv")

print(get_top_products(data))

print(get_top_products_by_shop(data, "Big Bite"))

print(get_trending_products(data))

print(get_peak_hours(data))

[{'item': 'Cigarettes Gold Flake', 'quantity': 15183}, {'item': 'Kurkure Masala', 'quantity': 15015}, {'item': 'Coca Cola 750ml', 'quantity': 14993}, {'item': 'Hide & Seek', 'quantity': 14926}, {'item': 'Pepsi 750ml', 'quantity': 14877}, {'item': 'Lays Magic Masala', 'quantity': 14845}, {'item': 'Cold Coffee Can', 'quantity': 14822}, {'item': 'Cup Noodles', 'quantity': 14813}, {'item': 'Red Bull', 'quantity': 14770}, {'item': 'Cigarettes Classic', 'quantity': 14761}]
[{'shop': 'Big Bite', 'item': 'Cold Coffee Can', 'quantity': 3772}, {'shop': 'Big Bite', 'item': 'Water Bottle 1L', 'quantity': 3756}, {'shop': 'Big Bite', 'item': 'Hide & Seek', 'quantity': 3707}, {'shop': 'Big Bite', 'item': 'Kurkure Masala', 'quantity': 3678}, {'shop': 'Big Bite', 'item': 'Coca Cola 750ml', 'quantity': 3659}]
[{'item': 'Oreo Biscuit', 'trend_score': 2.8}, {'item': 'Water Bottle 1L', 'trend_score': 2.2}, {'item': 'Red Bull', 'trend_score': 2.12}, {'item': 'Cup Noodles', 'trend_score': 2.12}, {'item': 'Ma

In [26]:
top_items = data['item'].value_counts().head(15).index
filtered_data = data[data['item'].isin(top_items)]

In [27]:
from mlxtend.frequent_patterns import apriori, association_rules

# Remove duplicates
filtered_data = filtered_data[['transaction_id', 'item']].drop_duplicates()

# Create basket
basket = (filtered_data
          .groupby(['transaction_id', 'item'])['item']
          .count()
          .unstack()
          .fillna(0))

# Convert to boolean
basket = basket > 0

# Apply Apriori
frequent_itemsets = apriori(
    basket,
    min_support=0.001,
    use_colnames=True
)

print("Frequent itemsets:", frequent_itemsets.shape)

# Generate rules
rules = association_rules(
    frequent_itemsets,
    metric="confidence",
    min_threshold=0.05
)

rules = rules.sort_values(by="lift", ascending=False)

print("\n🔹 Top Market Basket Rules:\n")
print(rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']].head(10))

Frequent itemsets: (120, 2)

🔹 Top Market Basket Rules:

             antecedents          consequents   support  confidence      lift
60    (Good Day Biscuit)    (Coca Cola 750ml)  0.008133    0.074611  0.677093
61     (Coca Cola 750ml)   (Good Day Biscuit)  0.008133    0.073811  0.677093
110  (Lays Magic Masala)        (Cup Noodles)  0.008119    0.073638  0.676327
111        (Cup Noodles)  (Lays Magic Masala)  0.008119    0.074566  0.676327
64      (Kurkure Masala)    (Coca Cola 750ml)  0.008089    0.073135  0.663701
65     (Coca Cola 750ml)     (Kurkure Masala)  0.008089    0.073409  0.663701
119    (Water Bottle 1L)        (Cup Noodles)  0.007823    0.072128  0.662465
118        (Cup Noodles)    (Water Bottle 1L)  0.007823    0.071855  0.662465
97            (Red Bull)    (Cold Coffee Can)  0.007794    0.071361  0.660160
96     (Cold Coffee Can)           (Red Bull)  0.007794    0.072102  0.660160


In [28]:
#function for MBA
def get_bought_together(product_name, rules_df, top_n=5):
    results = []

    for _, row in rules_df.iterrows():
        if product_name in row['antecedents']:
            for item in row['consequents']:
                results.append((item, row['confidence']))

    results = sorted(results, key=lambda x: x[1], reverse=True)

    seen = set()
    final = []
    for item, _ in results:
        if item not in seen:
            seen.add(item)
            final.append(item)

    return final[:top_n]

In [29]:
print(get_bought_together("Kurkure Masala",rules))

['Coca Cola 750ml', 'Pepsi 750ml', 'Cigarettes Gold Flake', 'Red Bull', 'Cup Noodles']
